In [6]:
!pip install "mlflow==2.22.4" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 63.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 20.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 88.6 MB/s eta 0:00:0000:0100:01
  Using cached alembic-1.17.2-py3-none-any.whl (248 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 87.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 87.4 MB/s eta 0:00:00:00:0100:01
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 7.1 MB/s eta 0:00:0000:0100:01m
  Using cached docker-7.1.0-py3-none-any.whl (147 kB)
  Using cached flask-3.1.2-py3-none-any.whl (103 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 74.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 94.1 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 86.2 MB/s eta 0

In [7]:
import os
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# 1) 设置 MLflow Tracking 地址
#   - 在 docker network 里面，mlflow 服务名就是 “mlflow”，端口 5000
mlflow.set_tracking_uri("http://mlflow:5000")

# 2) 设置 S3 / MinIO 的环境变量（和 docker-compose 里 mlflow 容器保持一致）
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"
os.environ["AWS_REGION"] = "us-east-1"

# 3) 选定（或创建）实验
mlflow.set_experiment("demo-from-jupyter")

# 4) 一个最简单的 demo 训练 + 记录
with mlflow.start_run(run_name="rf-demo"):
    db = load_diabetes()
    X_train, X_test, y_train, y_test = train_test_split(
        db.data, db.target, test_size=0.25, random_state=42
    )

    rf = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
    rf.fit(X_train, y_train)

    preds = rf.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = mse ** 0.5

    # log 参数 / metric / 模型
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(rf, artifact_path="model")

print("done, rmse =", rmse)


2025/12/06 13:46:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run rf-demo at: http://mlflow:5000/#/experiments/1/runs/4394e3009d3741ff9783be4a34b6ecec
🧪 View experiment at: http://mlflow:5000/#/experiments/1


MlflowException: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: '<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>
'

In [8]:
import mlflow
print(mlflow.__version__)

3.7.0
